# 04 · Race simulator

Logic is in `f1strat.simulator`, `f1strat.tyres` and `f1strat.race_events`;
see `docs/simulator.md`. Demo race: Spain 2022 (dry, not a case study).
Driver pace comes from the full-data fit, which includes hindsight about this
race; that is fine for a demo, and Phase 6 uses cutoff fits.

In [1]:
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd

from f1strat.clean import clean_laps
from f1strat.data import load_season
from f1strat.deg_model import THIN_FIT_PATH, load_fit
from f1strat.race_events import SimParams
from f1strat.simulator import (
    RaceState,
    actual_plan,
    build_field,
    car_model,
    field_search,
    late_restart_rounds,
    make_draws,
    observed_pit_losses,
    rival_plan_options,
    search,
    simulate_field,
)
from f1strat.tyres import reconstruct_inventories, sets_left, validate_against_race

logging.disable(logging.WARNING)
FIG_DIR = Path("../reports/figures")
fitted = load_fit(THIN_FIT_PATH)
raw = load_season(2022)
cleaned = clean_laps(raw)
params = SimParams.load()

## 1. Race-day tyre inventories

Reconstructed from FP1–FP3 and qualifying (`f1strat.tyres`), then checked
against the race: every set a car fitted in the race should exist in its
inventory. Mismatches are reported, not patched.

In [2]:
events = {6: "Spain", 13: "Hungary", 10: "Britain", 7: "Monaco"}
inventories, summary, mismatches = {}, [], []
for rnd, name in events.items():
    invs = reconstruct_inventories(2022, rnd)
    inventories[rnd] = invs
    bad = validate_against_race(invs, raw, rnd).assign(event=name)
    n = raw[(raw.RoundNumber == rnd) & raw.Compound.isin(["SOFT", "MEDIUM", "HARD"])] \
        .groupby(["Driver", "Stint"]).ngroups
    summary.append({"event": name, "dry race stints": n, "mismatches": len(bad),
                    "TyreLife missing in race data": int((bad["set"] == "TyreLife missing in race data").sum()),
                    "off by one lap": int((bad["off_by"].abs() == 1).sum())})
    mismatches.append(bad)
summary = pd.DataFrame(summary)
mismatches = pd.concat(mismatches, ignore_index=True)
mismatches.drop(columns="inventory").to_csv(FIG_DIR / "tyre_inventory_mismatches.csv", index=False)
(FIG_DIR / "tyre_inventory_validation.md").write_text(summary.to_markdown(index=False) + "\n")
summary

,event,dry race stints,mismatches,TyreLife missing in race data,off by one lap
0,Spain,73,6,0,4
1,Hungary,62,8,0,3
2,Britain,67,1,0,1
3,Monaco,39,15,11,0


In [3]:
mismatches.drop(columns="inventory")

,Driver,first_lap,compound,set,laps_needed,nearest_laps,off_by,event
0,HAM,2,SOFT,"used, TyreLife 3",2.0,3.0,1.0,Spain
1,MAG,1,SOFT,"used, TyreLife 3",2.0,3.0,1.0,Spain
2,MSC,1,SOFT,"used, TyreLife 3",2.0,3.0,1.0,Spain
3,RUS,52,SOFT,"used, TyreLife 3",2.0,3.0,1.0,Spain
4,STR,28,MEDIUM,"used, TyreLife 2",1.0,11.0,10.0,Spain
5,VET,43,MEDIUM,"used, TyreLife 2",1.0,11.0,10.0,Spain
6,GAS,47,SOFT,new,0.0,3.0,3.0,Hungary
7,PER,1,SOFT,new,0.0,3.0,3.0,Hungary
8,RIC,1,SOFT,"used, TyreLife 3",2.0,3.0,1.0,Hungary
9,STR,15,MEDIUM,"used, TyreLife 2",1.0,0.0,-1.0,Hungary


In [4]:
pd.DataFrame({d: [inv.summary()] for d, inv in inventories[6].items()}, index=["Spain race-day sets"]).T

,Spain race-day sets
ALB,"S new×2, S (4)×2, M new×2, H new×1"
ALO,"S new×2, S (3)×1, S (4)×1, M new×2, H new×1"
BOT,"S (4)×2, S (7)×1, M new×2, H new×1"
GAS,"S new×1, S (4)×3, S (7)×1, M new×1, H new×1"
HAM,"S (4)×3, M new×2, H new×1"
LAT,"S new×2, S (4)×1, S (7)×1, M new×2, H new×1"
LEC,"S new×1, S (3)×1, S (4)×1, M new×2, H new×1"
MAG,"S (4)×4, M new×1, H new×1"
MSC,"S (4)×4, M new×1, H new×1"
NOR,"S new×1, S (4)×4, M new×1, H new×1"


## 2. Single car (Part A rerun with the inventory)

Verstappen from the end of lap 1, alone against the clock, using his
reconstructed sets. Stratified shortlist: the best 3 plans for each stop count
and compound sequence.

In [5]:
state = RaceState.from_laps(raw, 6, lap=1, total_laps=66)
state = state.with_sets({d: sets_left(inventories[6][d], raw, 6, 1) for d in state.cars})
print("VER sets after lap 1:", state.sets["VER"])
rng = np.random.default_rng(42)
model = car_model(fitted, 6, "VER", state.cars["VER"].team, rng.permutation(1000), rng,
                  current_stint=None)
losses = observed_pit_losses(cleaned, 6)
draws = make_draws(model, state.laps_remaining, losses, max_stops=3, rng=rng)
actual = {d: actual_plan(raw, 6, d, 1) for d in ["VER", "PER", "RUS", "SAI", "BOT", "HAM"]}
start = time.time()
single = search(model, state, "VER", draws, max_stops=3, include=list(actual.values()))
print(f"{single.n_legal:,} legal plans scored, {len(single.plans)} in the stratified shortlist, "
      f"{time.time() - start:.2f} s")
tab = single.table(reference=actual["VER"])
tab["actual_by"] = tab["plan"].map(lambda lbl: ",".join(d for d, p in actual.items() if p.label("SOFT") == lbl))
cols = ["plan", "stops", "mean_s", "gap_to_best_s", "saved_vs_ref_s", "saved_p10_s",
        "saved_p90_s", "p_beats_ref", "actual_by"]
shown = pd.concat([tab.groupby("stops").head(3), tab[tab["actual_by"] != ""]]).drop_duplicates("plan")[cols]
(FIG_DIR / "spain_single_car.md").write_text(shown.round(2).to_markdown(index=False) + "\n")
shown.round(2)

VER sets after lap 1: (('HARD', 1), ('MEDIUM', 1), ('MEDIUM', 1), ('SOFT', 4), ('SOFT', 4))
764,192 legal plans scored, 87 in the stratified shortlist, 0.08 s


,plan,stops,mean_s,gap_to_best_s,saved_vs_ref_s,saved_p10_s,saved_p90_s,p_beats_ref,actual_by
0,S → 17 S(used) → 33 S(used) → 50 M(new),3,5820.96,0.00,5.36,0.80,10.19,0.94,
1,S → 17 M(new) → 33 S(used) → 50 S(used),3,5820.96,0.00,5.36,0.80,10.19,0.94,
2,S → 16 S(used) → 33 S(used) → 50 M(new),3,5820.97,0.01,5.35,0.75,10.05,0.95,
18,S → 22 M(new) → 44 S(used),2,5823.57,2.62,2.75,-5.36,10.92,0.67,
19,S → 22 S(used) → 44 M(new),2,5823.57,2.62,2.75,-5.36,10.92,0.67,
20,S → 22 M(new) → 45 S(used),2,5823.58,2.62,2.74,-5.51,10.89,0.66,
81,S → 32 M(new),1,5850.07,29.12,-23.75,-37.95,-9.83,0.02,
82,S → 33 M(new),1,5850.12,29.16,-23.80,-38.12,-9.80,0.02,
83,S → 31 M(new),1,5850.29,29.34,-23.97,-37.97,-10.14,0.02,
24,S → 17 M(new) → 37 M(new) → 53 S(used),3,5824.52,3.57,1.80,-3.04,7.00,0.67,PER


## 3. Whole field with traffic and safety cars (Part B)

- **Rivals:** each runs one of its own top-3 single-car plans, weighted by
  expected time (no hindsight).
- **Verstappen:** every plan in his stratified shortlist, simulated against
  the field.
- **Randomness:** SC/VSC at the pooled 2022 rates. Spain's own `final_race`
  comes from the posterior, because Spain had no late safety-car restart.

In [6]:
restart = late_restart_rounds(load_season(2022, table="race"))
print("races with a late SC restart (final_race drawn from the prior there):", restart)
start = time.time()
field = build_field(fitted, state, losses, params, n_sims=1000, seed=1,
                    use_fitted_current_stint=False, final_race_prior=6 in restart)
rivals = {d: rival_plan_options(field, d) for d in field.drivers if d != "VER"}
cands = search(field.models["VER"], state, "VER", field.draws["VER"], max_stops=3, min_stint=5,
               include=[actual["VER"]]).plans
setup = time.time() - start
start = time.time()
res = field_search(field, "VER", cands, rivals)
elapsed = time.time() - start
print(f"setup {setup:.1f} s; {len(cands)} plans x {field.n_sims} sims with the full field: "
      f"{elapsed:.1f} s ({elapsed / len(cands):.2f} s per plan)")

races with a late SC restart (final_race drawn from the prior there): [1, 5, 9, 10, 15, 17, 21]


setup 1.3 s; 82 plans x 1000 sims with the full field: 51.4 s (0.63 s per plan)


In [7]:
ftab = res.table(reference=actual["VER"])
ftab["actual"] = ftab["plan"] == actual["VER"].label("SOFT")
shown = pd.concat([ftab.groupby("stops").head(3), ftab[ftab["actual"]]]).drop_duplicates("plan")
(FIG_DIR / "spain_field.md").write_text(shown.round(3).to_markdown(index=False) + "\n")
shown.round(3)

,plan,stops,mean_pos,p_win,p_podium,p10_pos,p90_pos,mean_time_s,places_gained,p_ahead_of_ref,p_behind_ref,actual
0,S → 22 M(new) → 45 S(used),2,1.957,0.262,0.956,1.0,3.0,6054.505,0.239,0.216,0.092,False
1,S → 22 M(new) → 44 S(used),2,1.964,0.269,0.952,1.0,3.0,6054.622,0.232,0.206,0.088,False
2,S → 23 M(new) → 45 S(used),2,1.976,0.257,0.954,1.0,3.0,6054.763,0.220,0.208,0.092,False
7,S → 16 M(new) → 33 S(used) → 50 S(used),3,2.019,0.297,0.930,1.0,3.0,6053.632,0.177,0.267,0.105,False
9,S → 17 M(new) → 33 S(used) → 50 S(used),3,2.022,0.290,0.934,1.0,3.0,6053.656,0.174,0.258,0.102,False
10,S → 17 S(used) → 33 M(new) → 50 S(used),3,2.028,0.290,0.929,1.0,3.0,6053.729,0.168,0.258,0.106,False
76,S → 33 M(new),1,3.548,0.044,0.574,2.0,6.0,6078.014,-1.352,0.031,0.632,False
77,S → 32 M(new),1,3.577,0.045,0.574,2.0,6.0,6077.942,-1.381,0.034,0.645,False
78,S → 31 M(new),1,3.671,0.046,0.560,2.0,7.0,6078.113,-1.475,0.036,0.654,False
30,S → 13 M(new) → 28 S(used) → 44 M(new),3,2.196,0.204,0.906,1.0,3.0,6059.285,0.000,0.000,0.000,True


In [8]:
best = res.plans[int(np.argmin(res.positions[:, :, res.drivers.index("VER")].mean(axis=1)))]
fp = res.field_positions(best)
(FIG_DIR / "spain_field_positions.md").write_text(fp.round(2).to_markdown(index=False) + "\n")
print("Verstappen plan:", best.label("SOFT"))
fp.round(2)

Verstappen plan: S → 22 M(new) → 45 S(used)


,driver,mean_pos,p_win,p_podium,p10_pos,p90_pos
0,LEC,1.48,0.70,0.96,1.0,2.0
1,VER,1.96,0.26,0.96,1.0,3.0
2,PER,4.14,0.01,0.45,3.0,6.0
3,SAI,4.57,0.02,0.28,3.0,7.0
4,RUS,5.29,0.00,0.14,3.0,7.0
5,HAM,5.71,0.01,0.13,3.0,7.0
6,BOT,5.77,0.00,0.08,4.0,7.0
7,OCO,8.24,0.00,0.00,7.0,10.0
8,NOR,10.19,0.00,0.00,8.0,13.0
9,ALO,10.58,0.00,0.00,8.0,14.0


In [9]:
one = simulate_field(field, {**rivals, "VER": best})
n = field.n_sims
print(f"per simulated race: {one.attempts / n:.0f} pass attempts, {one.passes / n:.0f} passes, "
      f"{one.holds / n:.0f} laps held up; SC on {(field.status == 1).mean():.1%} of laps, "
      f"VSC on {(field.status == 2).mean():.1%}")

per simulated race: 179 pass attempts, 93 passes, 151 laps held up; SC on 6.2% of laps, VSC on 2.8%
